<div style="background: linear-gradient(135deg, #1A237E 0%, #3949AB 100%); padding: 30px; border-radius: 12px; text-align: center;">
  <h1 style="color: white; font-size: 2em; margin: 0;">🗞️ Moteur de Recommandation d'Articles</h1>
  <p style="color: #C5CAE9; font-size: 1.1em; margin-top: 10px;">Projet Natural Language Processing</p>
  <hr style="border-color: rgba(255,255,255,0.3); margin: 15px 0;">
  <p style="color: #E8EAF6; margin: 0;">
    <strong>Dataset :</strong> BBC News (~2225 articles, 5 catégories) &nbsp;|&nbsp;
    <strong>Approche 1 :</strong> TF-IDF + Cosine &nbsp;|&nbsp;
    <strong>Approche 2 :</strong> Sentence-BERT + FAISS
  </p>
</div>

---

## 📋 Table des matières
| Section | Contenu | Cours NLP |
|---------|---------|----------|
| **1** | Introduction & Problématique | — |
| **2** | Données : Chargement & Exploration | Cours 1 |
| **3** | Prétraitement NLP | Cours 1-2 |
| **4** | Approche 1 — TF-IDF + Cosinus (Baseline) | Cours 2 |
| **5** | Approche 2 — Sentence-BERT + FAISS (Avancée) | Cours 7-8 |
| **6** | Évaluation & Comparaison | Cours 9 |
| **7** | Discussion critique & Conclusion | — |

---
## 1. Introduction

### 1.1 Contexte et problématique
Face à l'explosion du volume de contenus numériques, les utilisateurs peinent à découvrir des articles pertinents. Un **moteur de recommandation basé sur la similarité textuelle** (*Content-Based Filtering*) permet de suggérer automatiquement des articles sémantiquement proches, sans dépendre de l'historique d'autres utilisateurs.

> **Question de recherche :** Les embeddings sémantiques (Sentence-BERT) améliorent-ils significativement la qualité des recommandations par rapport aux méthodes traditionnelles (TF-IDF) sur un corpus d'articles journalistiques ?

### 1.2 Objectifs
- Implémenter deux approches de recommandation par similarité textuelle
- Comparer TF-IDF (méthode classique) à Sentence-BERT (méthode neuronale)
- Évaluer quantitativement avec Precision@k et MRR
- Analyser les forces et limites de chaque approche

### 1.3 Ancrage dans le cours
| Technique | Cours | Module |
|-----------|-------|--------|
| Tokenisation, stemming, stopwords | Cours 1 | Module 1 |
| TF-IDF, similarité cosinus | Cours 2 | Module 1 |
| Loi de Zipf | Cours 1 | Module 1 |
| Transformers, Self-Attention | Cours 7 | Module 2 |
| SBERT, transfer learning | Cours 7-8 | Module 2-3 |
| FAISS, Precision@k, MRR | Cours 9 | Module 3 |

---
## 2. Installation & Imports

In [1]:
# ============================================================
# CELLULE 1 — Installation des dépendances
# ============================================================
# Décommenter et exécuter uniquement sur Google Colab
!pip install sentence-transformers faiss-cpu datasets nltk -q
print('✅ Packages installés avec succès')

✅ Packages installés avec succès



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ============================================================
# CELLULE 2 — Imports complets
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter
import re
import string
import time
import warnings
warnings.filterwarnings('ignore')

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# Scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE

# Sentence Transformers
from sentence_transformers import SentenceTransformer

# FAISS (recherche approximative — Cours 9)
try:
    import faiss
    FAISS_AVAILABLE = True
    print('✅ FAISS disponible')
except ImportError:
    FAISS_AVAILABLE = False
    print('⚠️  FAISS non disponible — fallback numpy cosine_similarity')

# Téléchargement ressources NLTK
for resource in ['punkt', 'stopwords', 'wordnet', 'punkt_tab']:
    nltk.download(resource, quiet=True)

# Configuration globale
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.titlesize': 13, 'axes.titleweight': 'bold'})

print('✅ Imports terminés')


✅ FAISS disponible
✅ Imports terminés


---
## 3. Données : Chargement & Exploration

In [ ]:
# ============================================================
# CELLULE 3 — Chargement du BBC News Dataset
# Option A (recommandée) : HuggingFace Datasets
# Option B : Kaggle CSV (décommenter si nécessaire)
# ============================================================
from datasets import load_dataset

print('⏳ Chargement du BBC News Dataset...')

# --- OPTION A : HuggingFace (défaut) ---
dataset = load_dataset('SetFit/bbc-news', split='train')
df = pd.DataFrame(dataset)

# --- OPTION B : Kaggle local (décommenter si besoin) ---
# df = pd.read_csv('bbc-news-data.csv', sep='\t')
# df.rename(columns={'category':'category','text':'text'}, inplace=True)

# Normalisation des noms de colonnes
col_map = {}
if 'label_text' in df.columns: col_map['label_text'] = 'category'
elif 'label' in df.columns:    col_map['label'] = 'category'
if col_map: df.rename(columns=col_map, inplace=True)

# Titre synthétique si absent
if 'title' not in df.columns:
    df['title'] = df['text'].apply(lambda x: ' '.join(str(x).split()[:10]) + '...')

# Index unique
df = df.reset_index(drop=True)
df['article_id'] = df.index

print(f'✅ Dataset chargé : {len(df)} articles')
print(f'   Colonnes : {list(df.columns)}')
df.head(3)

In [ ]:
# ============================================================
# CELLULE 4 — Analyse Exploratoire des Données (EDA)
# ============================================================
print('=' * 55)
print('ANALYSE EXPLORATOIRE')
print('=' * 55)

# Métriques de base
df['text_length'] = df['text'].apply(lambda x: len(str(x).split()))
category_counts   = df['category'].value_counts()

print(f'\nNombre total d\'articles : {len(df)}')
print(f'Catégories              : {list(df["category"].unique())}')
print(f'\nDistribution par catégorie :')
print(category_counts.to_string())
print(f'\nLongueur des textes (mots) :')
print(df['text_length'].describe().round(1).to_string())

# ── Visualisation ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#1A237E','#3949AB','#5C6BC0','#7986CB','#9FA8DA']

# 1. Distribution des catégories
axes[0].bar(category_counts.index, category_counts.values,
            color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('Distribution des catégories')
axes[0].set_xlabel('Catégorie')
axes[0].set_ylabel('Nombre d\'articles')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(category_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=10, fontweight='bold')

# 2. Distribution de la longueur
axes[1].hist(df['text_length'], bins=50, color='#3949AB',
             edgecolor='white', alpha=0.85)
axes[1].axvline(df['text_length'].mean(), color='#E65100',
                linestyle='--', linewidth=2, label=f'Moyenne : {df["text_length"].mean():.0f}')
axes[1].set_title('Distribution de la longueur')
axes[1].set_xlabel('Nombre de mots')
axes[1].set_ylabel('Fréquence')
axes[1].legend()

# 3. Longueur par catégorie (boxplot)
cat_order = category_counts.index.tolist()
data_by_cat = [df[df['category'] == c]['text_length'].values for c in cat_order]
bp = axes[2].boxplot(data_by_cat, labels=cat_order, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[2].set_title('Longueur par catégorie')
axes[2].set_xlabel('Catégorie')
axes[2].set_ylabel('Nombre de mots')
axes[2].tick_params(axis='x', rotation=30)

plt.suptitle('BBC News Dataset — Analyse Exploratoire', fontsize=15,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA terminée')

---
## 4. Prétraitement NLP

**Pipeline appliqué (Cours 1) :**
```
Texte brut → Minuscules → Nettoyage → Tokenisation → Stopwords → Stemming (Porter)
```
> **Note :** Le stemming est appliqué uniquement pour TF-IDF. Sentence-BERT utilise le texte brut (il gère nativement la tokenisation via son tokenizer WordPiece).

In [ ]:
# ============================================================
# CELLULE 5 — Classe TextPreprocessor
# ============================================================
class TextPreprocessor:
    """
    Pipeline complet de prétraitement NLP (Cours 1).
    Implémente : nettoyage, tokenisation, stopwords, stemming (Porter).
    """
    def __init__(self):
        self.stemmer    = PorterStemmer()
        self.stop_words = set(stopwords.words('english'))

    def clean(self, text):
        """Étape 1-2 : minuscules + suppression bruit"""
        text = str(text).lower()
        text = re.sub(r'http\S+|www\S+', '', text)        # URLs
        text = re.sub(r'@\w+|#\w+', '', text)             # mentions/hashtags
        text = text.translate(
            str.maketrans('', '', string.punctuation))     # ponctuation
        text = re.sub(r'\d+', '', text)                   # chiffres
        text = re.sub(r'\s+', ' ', text).strip()          # espaces multiples
        return text

    def preprocess(self, text, apply_stemming=True):
        """Pipeline complet : clean → tokenize → stopwords → [stem]"""
        cleaned = self.clean(text)
        tokens  = word_tokenize(cleaned)
        tokens  = [t for t in tokens
                   if t not in self.stop_words and len(t) > 2]
        if apply_stemming:
            tokens = [self.stemmer.stem(t) for t in tokens]
        return ' '.join(tokens)


# Application du pipeline
preprocessor = TextPreprocessor()

print('⏳ Application du prétraitement...')
# Texte stemmé → TF-IDF
df['text_stemmed'] = df['text'].apply(
    lambda x: preprocessor.preprocess(x, apply_stemming=True))
# Texte nettoyé (sans stemming) → SBERT
df['text_clean']   = df['text'].apply(
    lambda x: preprocessor.preprocess(x, apply_stemming=False))

# Affichage avant / après
idx = 10
print(f'\n{'='*55}')
print('EXEMPLE DE PRÉTRAITEMENT')
print(f'{'='*55}')
print(f'ORIGINAL  : {str(df["text"].iloc[idx])[:220]}...')
print(f'\nNETTOYÉ   : {df["text_clean"].iloc[idx][:220]}...')
print(f'\nSTEMMÉ    : {df["text_stemmed"].iloc[idx][:220]}...')
print(f'\n✅ Prétraitement terminé sur {len(df)} articles')

In [ ]:
# ============================================================
# CELLULE 6 — Analyse du vocabulaire & Loi de Zipf (Cours 1)
# ============================================================
all_words  = ' '.join(df['text_stemmed']).split()
word_freq  = Counter(all_words)
sorted_freq = sorted(word_freq.values(), reverse=True)

print(f'Vocabulaire avant prétraitement : {df["text"].apply(lambda x: len(str(x).split())).sum()} tokens')
print(f'Vocabulaire après prétraitement : {len(all_words)} tokens')
print(f'Mots uniques (après)            : {len(word_freq)}')
print(f'\nTop 15 mots :')
for w, c in word_freq.most_common(15):
    print(f'  {w:<20} {c}')

# Visualisation Zipf
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loi de Zipf (log-log)
ranks = range(1, len(sorted_freq) + 1)
axes[0].loglog(ranks, sorted_freq, color='#3949AB', linewidth=1.2, alpha=0.8)
axes[0].set_title('Loi de Zipf — BBC News Corpus')
axes[0].set_xlabel('Rang (échelle log)')
axes[0].set_ylabel('Fréquence (échelle log)')
axes[0].grid(True, alpha=0.3)
axes[0].annotate('Pente ≈ -1\n(loi de Zipf)', xy=(10, sorted_freq[9]),
                 xytext=(100, sorted_freq[9]*5),
                 arrowprops=dict(arrowstyle='->', color='#E65100'),
                 color='#E65100', fontsize=10)

# Réduction du vocabulaire
orig_lengths = df['text'].apply(lambda x: len(str(x).split()))
proc_lengths = df['text_stemmed'].apply(lambda x: len(x.split()))
axes[1].hist(orig_lengths, bins=40, alpha=0.6, color='#E65100', label='Original', edgecolor='white')
axes[1].hist(proc_lengths, bins=40, alpha=0.6, color='#3949AB', label='Après prétraitement', edgecolor='white')
axes[1].set_title('Réduction de dimension par prétraitement')
axes[1].set_xlabel('Nombre de tokens')
axes[1].set_ylabel('Fréquence')
axes[1].legend()

plt.suptitle('Analyse du Corpus — Loi de Zipf & Prétraitement', fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('corpus_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Approche 1 — TF-IDF + Similarité Cosinus (Baseline)

**Principe (Cours 2) :** Chaque article est représenté comme un vecteur TF-IDF dans un espace creux. La similarité entre deux articles est mesurée par le cosinus de l'angle entre leurs vecteurs.

$$w_{i,j} = tf_{i,j} \times \log\left(\frac{N}{n_i}\right) \qquad\qquad \cos(\vec{d_1}, \vec{d_2}) = \frac{\vec{d_1} \cdot \vec{d_2}}{\|\vec{d_1}\| \times \|\vec{d_2}\|}$$

In [ ]:
# ============================================================
# CELLULE 7 — Classe TFIDFRecommender
# ============================================================
class TFIDFRecommender:
    """
    Moteur de recommandation basé sur TF-IDF et similarité cosinus.
    Sémantique vectorielle classique — Cours 2.
    """
    def __init__(self, max_features=10000, ngram_range=(1, 2)):
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=ngram_range,   # unigrammes + bigrammes
            min_df=2,                  # ignorer hapax
            max_df=0.90,               # ignorer termes quasi-universels
            sublinear_tf=True          # log(1 + tf) pour réduire l'impact des hautes fréquences
        )
        self.tfidf_matrix = None
        self.articles_df  = None

    def fit(self, articles_df, text_column='text_stemmed'):
        """Construction de la matrice TF-IDF"""
        self.articles_df  = articles_df.reset_index(drop=True)
        self.tfidf_matrix = self.vectorizer.fit_transform(
            articles_df[text_column])
        print(f'  Dimensions TF-IDF : {self.tfidf_matrix.shape}')
        sparsity = 1 - self.tfidf_matrix.nnz / (
            self.tfidf_matrix.shape[0] * self.tfidf_matrix.shape[1])
        print(f'  Sparsité          : {sparsity:.4f} ({sparsity*100:.2f}%)')
        return self

    def recommend(self, article_idx, top_k=5, verbose=False):
        """Retourne les top_k articles les plus similaires"""
        query_vec = self.tfidf_matrix[article_idx]
        scores    = cosine_similarity(query_vec, self.tfidf_matrix).flatten()
        scores[article_idx] = -1  # exclure l'article source
        top_idx = scores.argsort()[::-1][:top_k]

        recs = []
        for idx in top_idx:
            recs.append({
                'index'           : int(idx),
                'title'           : self.articles_df.iloc[idx]['title'],
                'category'        : self.articles_df.iloc[idx]['category'],
                'similarity_score': float(scores[idx])
            })

        if verbose:
            src = self.articles_df.iloc[article_idx]
            print(f'📰 SOURCE [{src["category"].upper()}] : {src["title"]}')
            print('🔍 Recommandations TF-IDF :')
            for r, rec in enumerate(recs, 1):
                tag = '✅' if rec['category'] == src['category'] else '❌'
                print(f'  {r}. {tag} [{rec["category"]}] {rec["title"]} — score={rec["similarity_score"]:.4f}')
        return recs

    def get_top_terms(self, article_idx, n=10):
        """Retourne les termes TF-IDF les plus importants (interprétabilité)"""
        features = self.vectorizer.get_feature_names_out()
        vec      = self.tfidf_matrix[article_idx].toarray().flatten()
        top_idx  = vec.argsort()[-n:][::-1]
        return [(features[i], round(float(vec[i]), 4)) for i in top_idx]


# ── Entraînement ────────────────────────────────────────────
print('⏳ Entraînement TF-IDF...')
tfidf_rec = TFIDFRecommender(max_features=10000, ngram_range=(1, 2))
tfidf_rec.fit(df, text_column='text_stemmed')
print('✅ TF-IDF prêt')

In [ ]:
# ============================================================
# CELLULE 8 — Démonstration TF-IDF
# ============================================================
DEMO_IDX = 42

print('Termes TF-IDF les plus importants (interprétabilité) :')
for term, score in tfidf_rec.get_top_terms(DEMO_IDX, n=10):
    bar = '█' * int(score * 80)
    print(f'  {term:<25} {score:.4f}  {bar}')

print()
recs_tfidf_demo = tfidf_rec.recommend(DEMO_IDX, top_k=5, verbose=True)

---
## 6. Approche 2 — Sentence-BERT + FAISS (Avancée)

**Principe (Cours 7-8) :** SBERT utilise un Transformer encodeur bidirectionnel fine-tuné pour produire des embeddings denses capturant le sens sémantique en contexte.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Modèle : `all-MiniLM-L6-v2` — 6 couches, 384 dimensions, ~23M paramètres

In [ ]:
# ============================================================
# CELLULE 9 — Classe SBERTRecommender
# ============================================================
class SBERTRecommender:
    """
    Moteur de recommandation basé sur Sentence-BERT et FAISS.
    Embeddings sémantiques denses — Cours 7/8.
    Recherche approximative FAISS — Cours 9 (RAG Architecture).
    """
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        print(f'⏳ Chargement du modèle SBERT : {model_name}')
        self.model       = SentenceTransformer(model_name)
        self.emb_dim     = self.model.get_sentence_embedding_dimension()
        self.embeddings  = None
        self.faiss_index = None
        self.articles_df = None
        self.model_name  = model_name
        print(f'✅ Modèle chargé — dimension : {self.emb_dim}')

    def fit(self, articles_df, text_column='text_clean', batch_size=64):
        """Encodage + construction de l'index FAISS"""
        self.articles_df = articles_df.reset_index(drop=True)
        texts = articles_df[text_column].apply(
            lambda x: str(x)[:512]).tolist()  # troncature 512 tokens

        print(f'⏳ Encodage de {len(texts)} articles...')
        self.embeddings = self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True  # normalisation L2 → cosinus = produit scalaire
        )
        print(f'  Shape : {self.embeddings.shape}')
        self._build_faiss_index()
        return self

    def _build_faiss_index(self):
        """Index FAISS IndexFlatIP (produit scalaire = cosinus après norm. L2)"""
        if FAISS_AVAILABLE:
            self.faiss_index = faiss.IndexFlatIP(self.emb_dim)
            self.faiss_index.add(self.embeddings.astype('float32'))
            print(f'  Index FAISS : {self.faiss_index.ntotal} vecteurs indexés')
        else:
            print('  Fallback : numpy cosine_similarity')

    def recommend(self, article_idx, top_k=5, verbose=False):
        """Retourne les top_k articles les plus similaires"""
        query_emb = self.embeddings[article_idx].reshape(1, -1).astype('float32')

        if FAISS_AVAILABLE and self.faiss_index:
            scores, indices = self.faiss_index.search(query_emb, top_k + 5)
            scores, indices = scores.flatten(), indices.flatten()
        else:
            sims    = cosine_similarity(query_emb, self.embeddings).flatten()
            indices = sims.argsort()[::-1]
            scores  = sims[indices]

        recs = []
        for idx, score in zip(indices, scores):
            if idx == article_idx: continue
            recs.append({
                'index'           : int(idx),
                'title'           : self.articles_df.iloc[idx]['title'],
                'category'        : self.articles_df.iloc[idx]['category'],
                'similarity_score': float(score)
            })
            if len(recs) >= top_k: break

        if verbose:
            src = self.articles_df.iloc[article_idx]
            print(f'📰 SOURCE [{src["category"].upper()}] : {src["title"]}')
            print('🔍 Recommandations SBERT :')
            for r, rec in enumerate(recs, 1):
                tag = '✅' if rec['category'] == src['category'] else '❌'
                print(f'  {r}. {tag} [{rec["category"]}] {rec["title"]} — score={rec["similarity_score"]:.4f}')
        return recs

    def find_similar_by_query(self, query_text, top_k=5):
        """Recherche par texte libre (pas d'article existant requis)"""
        q_emb = self.model.encode([query_text],
                                   normalize_embeddings=True,
                                   convert_to_numpy=True).astype('float32')
        if FAISS_AVAILABLE and self.faiss_index:
            scores, indices = self.faiss_index.search(q_emb, top_k)
            scores, indices = scores.flatten(), indices.flatten()
        else:
            sims    = cosine_similarity(q_emb, self.embeddings).flatten()
            indices = sims.argsort()[::-1][:top_k]
            scores  = sims[indices]
        return [{'index': int(i),
                 'title': self.articles_df.iloc[i]['title'],
                 'category': self.articles_df.iloc[i]['category'],
                 'similarity_score': float(s)}
                for i, s in zip(indices, scores)]


# ── Entraînement ────────────────────────────────────────────
sbert_rec = SBERTRecommender(model_name='all-MiniLM-L6-v2')
sbert_rec.fit(df, text_column='text_clean', batch_size=64)
print('\n✅ SBERT prêt')

In [ ]:
# ============================================================
# CELLULE 10 — Démonstration SBERT (même article que TF-IDF)
# ============================================================
print('=== COMPARAISON SUR UN MÊME ARTICLE ===')
print()

print('--- TF-IDF ---')
_ = tfidf_rec.recommend(DEMO_IDX, top_k=5, verbose=True)

print()
print('--- SBERT ---')
_ = sbert_rec.recommend(DEMO_IDX, top_k=5, verbose=True)

# Bonus : recherche par texte libre (SBERT uniquement)
print()
print('=== RECHERCHE PAR TEXTE LIBRE (SBERT) ===')
query = 'artificial intelligence and machine learning in healthcare'
print(f'Requête : "{query}"')
results = sbert_rec.find_similar_by_query(query, top_k=4)
for i, r in enumerate(results, 1):
    print(f'  {i}. [{r["category"]}] {r["title"]} — score={r["similarity_score"]:.4f}')

In [ ]:
# ============================================================
# CELLULE 11 — Visualisation t-SNE des embeddings SBERT
# ============================================================
print('⏳ Calcul t-SNE (peut prendre ~1 min)...')

SAMPLE_SIZE = min(600, len(df))
sample_idx  = np.random.choice(len(df), SAMPLE_SIZE, replace=False)
emb_sample  = sbert_rec.embeddings[sample_idx]
cat_sample  = df['category'].iloc[sample_idx].values

tsne_2d = TSNE(n_components=2, perplexity=35,
               random_state=RANDOM_STATE, n_iter=1000).fit_transform(emb_sample)

fig, ax = plt.subplots(figsize=(11, 8))
palette = {'business':'#1A237E','entertainment':'#E65100',
           'politics':'#2E7D32','sport':'#C62828','tech':'#6A1B9A'}

for cat, color in palette.items():
    mask = cat_sample == cat
    ax.scatter(tsne_2d[mask, 0], tsne_2d[mask, 1],
               c=color, label=cat, alpha=0.65, s=22, edgecolors='none')

ax.set_title('Visualisation t-SNE des embeddings Sentence-BERT\n'
             f'BBC News Dataset — {SAMPLE_SIZE} articles', fontsize=14)
ax.legend(title='Catégorie', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_xlabel('t-SNE dim 1')
ax.set_ylabel('t-SNE dim 2')
ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.savefig('tsne_sbert.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ t-SNE : les clusters sémantiques sont bien séparés')

---
## 7. Évaluation & Comparaison

**Protocole :** En l'absence de labels explicites, la **catégorie** est utilisée comme proxy de pertinence. Une recommandation est pertinente si elle appartient à la même catégorie que l'article source.

$$\text{P@k} = \frac{|\{i \in \text{top-}k \mid \text{cat}(i) = \text{cat}_{\text{source}}\}|}{k} \qquad \text{MRR} = \frac{1}{|Q|}\sum_{q=1}^{|Q|} \frac{1}{\text{rang}_q}$$

In [ ]:
# ============================================================
# CELLULE 12 — Fonctions de métriques
# ============================================================
def precision_at_k(recs, source_category, k):
    """P@k : proportion de recommandations pertinentes dans le top-k."""
    relevant = sum(1 for r in recs[:k]
                   if r['category'] == source_category)
    return relevant / k

def mean_reciprocal_rank(recs, source_category):
    """MRR : inverse du rang du 1er résultat pertinent."""
    for rank, r in enumerate(recs, 1):
        if r['category'] == source_category:
            return 1.0 / rank
    return 0.0

def category_diversity(recs):
    """Diversité : proportion de catégories distinctes recommandées."""
    if not recs: return 0.0
    return len(set(r['category'] for r in recs)) / len(recs)

print('✅ Métriques définies : Precision@k, MRR, Diversity')

In [ ]:
# ============================================================
# CELLULE 13 — Évaluation systématique (échantillon stratifié)
# ============================================================
N_EVAL_PER_CAT = 10  # articles par catégorie → 50 au total
K_VALUES       = [3, 5, 10]

# Sélection stratifiée
eval_indices = []
for cat in df['category'].unique():
    cat_idx = df[df['category'] == cat].index.tolist()
    chosen  = np.random.choice(cat_idx,
                               size=min(N_EVAL_PER_CAT, len(cat_idx)),
                               replace=False)
    eval_indices.extend(chosen)

print(f'Évaluation sur {len(eval_indices)} articles (stratifié par catégorie)...')

# Stockage des résultats
metrics = {
    'tfidf': {f'p@{k}': [] for k in K_VALUES},
    'sbert': {f'p@{k}': [] for k in K_VALUES}
}
for m in metrics:
    metrics[m].update({'mrr': [], 'diversity': [], 'time_ms': []})

# Évaluation
for idx in eval_indices:
    src_cat = df.iloc[idx]['category']

    # TF-IDF
    t0 = time.time()
    recs_tfidf = tfidf_rec.recommend(idx, top_k=max(K_VALUES))
    dt_tfidf   = (time.time() - t0) * 1000

    # SBERT
    t0 = time.time()
    recs_sbert = sbert_rec.recommend(idx, top_k=max(K_VALUES))
    dt_sbert   = (time.time() - t0) * 1000

    for k in K_VALUES:
        metrics['tfidf'][f'p@{k}'].append(precision_at_k(recs_tfidf, src_cat, k))
        metrics['sbert'][f'p@{k}'].append(precision_at_k(recs_sbert, src_cat, k))

    metrics['tfidf']['mrr'].append(mean_reciprocal_rank(recs_tfidf, src_cat))
    metrics['sbert']['mrr'].append(mean_reciprocal_rank(recs_sbert, src_cat))
    metrics['tfidf']['diversity'].append(category_diversity(recs_tfidf))
    metrics['sbert']['diversity'].append(category_diversity(recs_sbert))
    metrics['tfidf']['time_ms'].append(dt_tfidf)
    metrics['sbert']['time_ms'].append(dt_sbert)

# Tableau récapitulatif
print('\n' + '=' * 55)
print('RÉSULTATS QUANTITATIFS (moyenne ± écart-type)')
print('=' * 55)
print(f'{"Métrique":<15} {"TF-IDF":>22} {"SBERT":>22} {"Gain":>10}')
print('-' * 72)
metric_keys = [f'p@{k}' for k in K_VALUES] + ['mrr', 'diversity', 'time_ms']
for mk in metric_keys:
    tf_m = np.mean(metrics['tfidf'][mk])
    tf_s = np.std(metrics['tfidf'][mk])
    sb_m = np.mean(metrics['sbert'][mk])
    sb_s = np.std(metrics['sbert'][mk])
    gain = f'{((sb_m/tf_m)-1)*100:+.1f}%' if tf_m > 0 else 'N/A'
    unit = ' ms' if mk == 'time_ms' else ''
    print(f'{mk:<15} {tf_m:>8.4f}±{tf_s:.4f}{unit}  '
          f'{sb_m:>8.4f}±{sb_s:.4f}{unit}  {gain:>10}')

In [ ]:
# ============================================================
# CELLULE 14 — Évaluation par catégorie
# ============================================================
cat_metrics = {cat: {'tfidf_p5': [], 'sbert_p5': []}
               for cat in df['category'].unique()}

for idx in eval_indices:
    cat = df.iloc[idx]['category']
    r_tf = tfidf_rec.recommend(idx, top_k=5)
    r_sb = sbert_rec.recommend(idx, top_k=5)
    cat_metrics[cat]['tfidf_p5'].append(precision_at_k(r_tf, cat, 5))
    cat_metrics[cat]['sbert_p5'].append(precision_at_k(r_sb, cat, 5))

print(f'{"Catégorie":<18} {"TF-IDF P@5":>12} {"SBERT P@5":>12} {"Gain":>8}')
print('-' * 54)
for cat in sorted(cat_metrics):
    tf_v = np.mean(cat_metrics[cat]['tfidf_p5'])
    sb_v = np.mean(cat_metrics[cat]['sbert_p5'])
    gain = f'{((sb_v/tf_v)-1)*100:+.1f}%' if tf_v > 0 else 'N/A'
    print(f'{cat:<18} {tf_v:>12.4f} {sb_v:>12.4f} {gain:>8}')

In [ ]:
# ============================================================
# CELLULE 15 — Visualisations comparatives complètes
# ============================================================
fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

C_TF = '#1A237E'
C_SB = '#E65100'

def add_value_labels(ax, bars, fmt='.3f'):
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.01,
                f'{h:{fmt}}', ha='center', va='bottom', fontsize=9)

# ── 1. Precision@k ──────────────────────────────────────────
ax1  = fig.add_subplot(gs[0, 0])
xlbl = [f'P@{k}' for k in K_VALUES]
x    = np.arange(len(K_VALUES))
w    = 0.35
b1   = ax1.bar(x - w/2, [np.mean(metrics['tfidf'][f'p@{k}']) for k in K_VALUES],
               w, label='TF-IDF', color=C_TF, edgecolor='white')
b2   = ax1.bar(x + w/2, [np.mean(metrics['sbert'][f'p@{k}']) for k in K_VALUES],
               w, label='SBERT',  color=C_SB, edgecolor='white')
add_value_labels(ax1, b1)
add_value_labels(ax1, b2)
ax1.set_title('Precision@k')
ax1.set_xticks(x); ax1.set_xticklabels(xlbl)
ax1.set_ylim(0, 1.15)
ax1.legend(); ax1.grid(axis='y', alpha=0.3)

# ── 2. MRR ──────────────────────────────────────────────────
ax2  = fig.add_subplot(gs[0, 1])
mrr_vals = [np.mean(metrics[m]['mrr']) for m in ['tfidf', 'sbert']]
bars = ax2.bar(['TF-IDF', 'SBERT'], mrr_vals,
               color=[C_TF, C_SB], edgecolor='white', width=0.5)
add_value_labels(ax2, bars, fmt='.4f')
ax2.set_title('Mean Reciprocal Rank (MRR)')
ax2.set_ylim(0, 1.15)
ax2.grid(axis='y', alpha=0.3)

# ── 3. Diversité ────────────────────────────────────────────
ax3  = fig.add_subplot(gs[0, 2])
div_vals = [np.mean(metrics[m]['diversity']) for m in ['tfidf', 'sbert']]
bars = ax3.bar(['TF-IDF', 'SBERT'], div_vals,
               color=[C_TF, C_SB], edgecolor='white', width=0.5)
add_value_labels(ax3, bars)
ax3.set_title('Diversité des recommandations')
ax3.set_ylim(0, 1.15)
ax3.grid(axis='y', alpha=0.3)

# ── 4. Precision@5 par catégorie ────────────────────────────
ax4 = fig.add_subplot(gs[1, :2])
cats   = sorted(cat_metrics.keys())
xc     = np.arange(len(cats))
bc1    = ax4.bar(xc - w/2, [np.mean(cat_metrics[c]['tfidf_p5']) for c in cats],
                 w, label='TF-IDF', color=C_TF, edgecolor='white')
bc2    = ax4.bar(xc + w/2, [np.mean(cat_metrics[c]['sbert_p5']) for c in cats],
                 w, label='SBERT',  color=C_SB, edgecolor='white')
add_value_labels(ax4, bc1)
add_value_labels(ax4, bc2)
ax4.set_title('Precision@5 par catégorie')
ax4.set_xticks(xc); ax4.set_xticklabels(cats, rotation=20)
ax4.set_ylim(0, 1.2)
ax4.legend(); ax4.grid(axis='y', alpha=0.3)

# ── 5. Temps d'inférence (log) ───────────────────────────────
ax5   = fig.add_subplot(gs[1, 2])
t_vals = [np.mean(metrics[m]['time_ms']) for m in ['tfidf', 'sbert']]
bars   = ax5.bar(['TF-IDF', 'SBERT'], t_vals,
                  color=[C_TF, C_SB], edgecolor='white', width=0.5)
for bar, v in zip(bars, t_vals):
    ax5.text(bar.get_x() + bar.get_width()/2., v * 1.05,
             f'{v:.2f} ms', ha='center', va='bottom', fontsize=10)
ax5.set_title('Temps d\'inférence moyen')
ax5.set_ylabel('Millisecondes (log)')
ax5.set_yscale('log')
ax5.grid(axis='y', alpha=0.3)

fig.suptitle('Évaluation comparative — TF-IDF vs Sentence-BERT\nBBC News Dataset',
             fontsize=16, fontweight='bold', y=1.01)
plt.savefig('evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualisations sauvegardées')

In [ ]:
# ============================================================
# CELLULE 16 — Heatmap similarité inter-catégories (SBERT)
# ============================================================
categories = sorted(df['category'].unique())
n_cat      = len(categories)
sim_mat    = np.zeros((n_cat, n_cat))
N_SAMPLE   = 40

for i, ci in enumerate(categories):
    for j, cj in enumerate(categories):
        embs_i = sbert_rec.embeddings[df['category'] == ci][:N_SAMPLE]
        embs_j = sbert_rec.embeddings[df['category'] == cj][:N_SAMPLE]
        sim_mat[i, j] = cosine_similarity(embs_i, embs_j).mean()

plt.figure(figsize=(8, 6))
sns.heatmap(sim_mat,
            xticklabels=categories, yticklabels=categories,
            annot=True, fmt='.3f', cmap='Blues',
            vmin=0.2, vmax=1.0,
            linewidths=0.5, linecolor='white')
plt.title('Similarité cosinus moyenne inter-catégories\n(Sentence-BERT)', fontsize=13)
plt.tight_layout()
plt.savefig('category_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Heatmap générée')

---
## 8. Discussion critique & Conclusion

In [ ]:
# ============================================================
# CELLULE 17 — Résumé exécutif automatique
# ============================================================
tf_p5 = np.mean(metrics['tfidf']['p@5'])
sb_p5 = np.mean(metrics['sbert']['p@5'])
gain  = (sb_p5 / tf_p5 - 1) * 100 if tf_p5 > 0 else 0

print('=' * 60)
print('RÉSUMÉ EXÉCUTIF — RÉSULTATS DU PROJET')
print('=' * 60)
print(f'\nDataset     : BBC News — {len(df)} articles, 5 catégories')
print(f'\nMétrique principale (Precision@5) :')
print(f'  TF-IDF       : {tf_p5:.4f}')
print(f'  Sentence-BERT: {sb_p5:.4f}')
print(f'  Gain SBERT   : {gain:+.1f}%')

print(f'\nMRR :')
print(f'  TF-IDF       : {np.mean(metrics["tfidf"]["mrr"]):.4f}')
print(f'  Sentence-BERT: {np.mean(metrics["sbert"]["mrr"]):.4f}')

print(f'\nTemps d\'inférence :')
print(f'  TF-IDF       : {np.mean(metrics["tfidf"]["time_ms"]):.2f} ms')
print(f'  Sentence-BERT: {np.mean(metrics["sbert"]["time_ms"]):.2f} ms')

print()
print('CONCLUSION :')
print(f'  SBERT surpasse TF-IDF de {gain:+.1f}% en Precision@5.')
print('  TF-IDF reste préférable quand la latence ou les ressources')
print('  computationnelles sont très contraintes.')
print()
print('TECHNIQUES DU COURS APPLIQUÉES :')
print('  ✅ Cours 1  : Prétraitement, Loi de Zipf')
print('  ✅ Cours 2  : TF-IDF, similarité cosinus')
print('  ✅ Cours 7  : Transformers, Self-Attention, BERT')
print('  ✅ Cours 8  : Transfer learning, modèles pré-entraînés')
print('  ✅ Cours 9  : FAISS (ANN), Precision@k, MRR')

### 8.1 Discussion critique

**Forces et limites comparées :**

| Critère | TF-IDF | Sentence-BERT |
|---------|--------|---------------|
| Précision sémantique | Moyenne | **Élevée** |
| Interprétabilité | **Élevée** (termes visibles) | Faible (boîte noire) |
| Vitesse d'inférence | **Très rapide** | Rapide (avec FAISS) |
| Ressources | **CPU suffisant** | GPU recommandé |
| Gestion des paraphrases | Faible | **Excellente** |
| Passage à l'échelle | **Excellent** (sparse) | Nécessite FAISS |
| Multilingue | Difficile | **Oui** (XLM-R, mBERT) |

**Limites de l'évaluation :**
- La catégorie comme proxy de pertinence est imparfaite (sport tennis ≠ sport football)
- Absence de feedback utilisateur réel
- Troncature SBERT à 512 tokens peut faire perdre des informations

### 8.2 Pistes d'amélioration
1. **Fine-tuning SBERT** sur le domaine BBC News (Cours 8 — SFT)
2. **Approche hybride** : $\alpha \cdot \text{TF-IDF} + (1-\alpha) \cdot \text{SBERT}$ avec RRF (Cours 9)
3. **Diversité MMR** (Maximal Marginal Relevance) : $\lambda \cdot \text{sim}(q,d) - (1-\lambda) \cdot \max_{s \in S}\text{sim}(d,s)$
4. **Re-ranking cross-encoder** (ColBERT, BGE-reranker) sur le top-100 (Cours 9)
5. **Déploiement** : API FastAPI + index FAISS persistant

In [ ]:
# ============================================================
# CELLULE 18 — Sauvegarde des résultats
# ============================================================
import pickle

# Export CSV des métriques
results_export = pd.DataFrame({
    'methode' : ['TF-IDF', 'SBERT'],
    'P@3'     : [np.mean(metrics[m]['p@3'])  for m in ['tfidf','sbert']],
    'P@5'     : [np.mean(metrics[m]['p@5'])  for m in ['tfidf','sbert']],
    'P@10'    : [np.mean(metrics[m]['p@10']) for m in ['tfidf','sbert']],
    'MRR'     : [np.mean(metrics[m]['mrr'])  for m in ['tfidf','sbert']],
    'Diversity': [np.mean(metrics[m]['diversity']) for m in ['tfidf','sbert']],
    'Time_ms' : [np.mean(metrics[m]['time_ms'])    for m in ['tfidf','sbert']],
})
results_export.to_csv('evaluation_results.csv', index=False)

# Sauvegarde des modèles
with open('tfidf_model.pkl', 'wb') as f:
    pickle.dump({'vectorizer'  : tfidf_rec.vectorizer,
                 'tfidf_matrix': tfidf_rec.tfidf_matrix}, f)

np.save('sbert_embeddings.npy', sbert_rec.embeddings)

print('✅ Fichiers sauvegardés :')
print('   - evaluation_results.csv')
print('   - tfidf_model.pkl')
print('   - sbert_embeddings.npy')
print('   - eda_overview.png')
print('   - corpus_analysis.png')
print('   - tsne_sbert.png')
print('   - evaluation_results.png')
print('   - category_heatmap.png')
print()
print('🎉 PROJET NLP — MOTEUR DE RECOMMANDATION TERMINÉ !')